In [1]:
import tensorflow as tf
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import cv2
import os
from PIL import Image

C:\Users\thaim\anaconda3\envs\tf210\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [2]:
def preprocess(
    img: np.ndarray,
    IMG_SIZE: int = 224,
    sigmaX: float = None,
    clahe_clip: float = 2.0,
    clahe_grid: tuple = (8,8),
    gain: float = 2.0,
    background_value: int = 0
) -> np.ndarray:
    """
    1) Find the “retina” circle (gray-threshold → largest contour → min-enclosing circle).
    2) Mask out (zero-out) everything outside that circle, then crop to the bounding square.
    3) Resize to (IMG_SIZE×IMG_SIZE).
    4) Do your unsharp mask / CLAHE steps on that cropped‐and‐resized ROI.
    5) Build a final circular mask (same size IMG_SIZE×IMG_SIZE) so that:
         - pixels INSIDE the circle come from your preprocessed image,
         - pixels OUTSIDE the circle become `background_value`.
    6) Return that final composite.
    """
    # ——————————————————————————
    # STEP A: find circle in original image
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    _, thresh = cv2.threshold(gray, 10, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # If we found at least one contour, pick the largest.
    if len(contours) > 0:
        cnt = max(contours, key=cv2.contourArea)
        (x, y), r = cv2.minEnclosingCircle(cnt)
        x, y, r = int(x), int(y), int(r)

        # Build a single-channel mask and zero out everything outside that circle:
        mask0 = np.zeros_like(gray, dtype=np.uint8)
        cv2.circle(mask0, (x, y), r, 255, thickness=-1)
        img_masked = cv2.bitwise_and(img, img, mask=mask0)

        # Crop to the square bounding box around that circle:
        x1, y1 = max(0, x - r), max(0, y - r)
        x2, y2 = min(img.shape[1], x + r), min(img.shape[0], y + r)
        roi = img_masked[y1:y2, x1:x2]
    else:
        # If no contour found, just fall back to using the entire image
        roi = img.copy()

    # ——————————————————————————
    # STEP B: resize the ROI to IMG_SIZE × IMG_SIZE
    roi_resized = cv2.resize(roi, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)

    # ——————————————————————————
    # STEP C: unsharp‐mask / background removal
    if sigmaX is None:
        sigmaX = 0.1 * IMG_SIZE
    blur = cv2.GaussianBlur(roi_resized, (0,0), sigmaX=sigmaX)
    sharpened = cv2.addWeighted(roi_resized, gain, blur, -gain, 128)

    # ——————————————————————————
    # STEP D: (optional) CLAHE on the green channel
    lab = cv2.cvtColor(sharpened, cv2.COLOR_RGB2LAB)
    L, A, B = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=clahe_clip, tileGridSize=clahe_grid)
    L2 = clahe.apply(L)
    lab2 = cv2.merge((L2, A, B))
    final_roi = cv2.cvtColor(lab2, cv2.COLOR_LAB2RGB)

    # ——————————————————————————
    # STEP E: build a circular mask of size (IMG_SIZE × IMG_SIZE)
    # (We know that after cropping → resizing to IMG_SIZE, the circle should now
    #   fill most of that square.  In fact, the circle’s center is (IMG_SIZE/2, IMG_SIZE/2)
    #   and radius = IMG_SIZE/2.)
    h, w = IMG_SIZE, IMG_SIZE
    circ_mask = np.zeros((h, w), dtype=np.uint8)
    center = (w // 2, h // 2)
    radius = w // 2
    cv2.circle(circ_mask, center, radius, 1, thickness=-1)  # mask = 1 inside circle, 0 outside

    # Convert mask to 3-channel so we can multiply with RGB:
    mask_3ch = np.stack([circ_mask]*3, axis=-1)

    # ——————————————————————————
    # STEP F: composite final image:
    #   - wherever mask_3ch == 1, use final_roi’s pixel;
    #   - wherever mask_3ch == 0, use background_value (e.g. 0 for black).
    out = np.full((h, w, 3), background_value, dtype=np.uint8)
    out[mask_3ch == 1] = final_roi[mask_3ch == 1]

    return out


In [3]:
def get_efficientNetB3(
    input_size: int = 224,
    num_classes: int = 5,
    backbone_trainable: bool = False,
    dropout_rate: float = 0.5,
    l2_reg: float = 0.01,
    l1_activity_reg: float = 0.002,
    learning_rate: float = 1e-3,
):
    """
    Creates a transfer‐learning model for diabetic‐retinopathy classification
    based on EfficientNetB3 (pretrained on ImageNet). By default, the EfficientNet
    backbone is frozen (backbone_trainable=False). After initial training you can
    unfreeze a few top layers.

    Args:
      input_size:          Height & width of the input (they should be square).
      num_classes:         Number of output classes (e.g. 5 for DR grades 0–4).
      backbone_trainable:  If False, the ImageNet‐trained EfficientNetB3 layers
                           stay frozen. If True, all of them become trainable.
      dropout_rate:        Dropout rate applied just before the final dense.
      l2_reg:              L2 weight regularizer strength for the final Dense.
      l1_activity_reg:     L1 activity regularizer strength for the final Dense.
      learning_rate:       Learning rate for the Adamax optimizer.

    Returns:
      A compiled tf.keras.Model ready for fit(…).
    """

    # 1) Build the EfficientNetB3 base (exclude top, ImageNet weights)
    base = tf.keras.applications.EfficientNetB3(
        include_top=False,
        weights="imagenet",
        input_shape=(input_size, input_size, 3),
        pooling="avg",
    )
    base.trainable = backbone_trainable

    # 2) Assemble the “head” on top of the frozen/finetunable backbone:
    inputs = tf.keras.Input(shape=(input_size, input_size, 3))

    # → Preprocessing (scales to ImageNet range)
    x = tf.keras.applications.efficientnet.preprocess_input(inputs)

    # → Backbone features
    x = base(x)

    # → BatchNorm (helps stabilize)
    x = tf.keras.layers.BatchNormalization(momentum=0.99, epsilon=1e-3)(x)

    # → Dropout for regularization
    x = tf.keras.layers.Dropout(dropout_rate)(x)

    # → Final Dense with both L2 on weights and L1 on activations
    x = tf.keras.layers.Dense(
        units=num_classes,
        activation="softmax",
        kernel_regularizer=tf.keras.regularizers.l2(l2_reg),
        activity_regularizer=tf.keras.regularizers.l1(l1_activity_reg),
    )(x)

    model = tf.keras.Model(inputs, x, name="EffNetB3_transfer")

    # 3) Compile
    model.compile(
        optimizer=tf.keras.optimizers.Adamax(learning_rate=learning_rate),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )

    return model


In [ ]:
def get_prediction(model,image):
    